In [1]:
# --- 导入库 ---
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import LabelEncoder

# --- 设置路径 ---
DATA_PATH = 'D:/pycharmProject/TimeSequences/data/'
OUTPUT_PATH = DATA_PATH  # 结果保存到同一目录

# --- 加载数据 ---
def load_data():
    # 核心数据
    train = pd.read_csv(DATA_PATH + 'train.csv', parse_dates=['date'])
    test = pd.read_csv(DATA_PATH + 'test.csv', parse_dates=['date'])

    # 外部数据
    holidays = pd.read_csv(DATA_PATH + 'holidays_events.csv', parse_dates=['date'])
    oil = pd.read_csv(DATA_PATH + 'oil.csv', parse_dates=['date'])
    stores = pd.read_csv(DATA_PATH + 'stores.csv')
    transactions = pd.read_csv(DATA_PATH + 'transactions.csv', parse_dates=['date'])

    return train, test, holidays, oil, stores, transactions

# --- 特征工程函数 ---
def create_date_features(df):
    #该函数的作用是 从日期列（date）中提取和构造多个时间相关特征，将日期信息分解为更细粒度或更具业务意义的特征（如月份、周数、是否周末等），帮助机器学习模型捕捉时间模式（如季节性、周期性、节假日效应等）。
    df['day'] = df['date'].dt.day
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype('int')
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['quarter'] = df['date'].dt.quarter  # 新增季度特征
    return df

def create_lag_features(df, lags=[1, 7, 14, 28]):
    #该函数的目的是 为时间序列数据创建滞后特征（lag features），即用历史时间点的值（如过去 1 天、7 天的销售额）作为当前时间点的特征。这类特征常用于时间序列预测模型（如销量预测），帮助模型捕捉历史趋势、周期性和依赖性
    for lag in lags:
        df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
    return df

def create_rolling_features(df, windows=[7, 14, 28]):
    #该函数用于为时间序列数据创建滚动窗口统计特征（如滚动均值），通过历史数据的平滑处理捕捉趋势和周期性模式。其核心逻辑是：对每个商店（store_nbr）和商品类别（family）的分组数据，计算滞后一步（避免数据泄露）后，生成指定窗口大小的滚动均值特征。
    for window in windows:
        df[f'sales_roll_mean_{window}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(1).rolling(window=window).mean()
    return df

# --- 数据合并与预处理 ---
def merge_features(train, test, holidays, oil, stores, transactions):
    # 合并油价数据
    oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
    full_data = pd.concat([train, test], ignore_index=True)
    full_data = pd.merge(full_data, oil, on='date', how='left')

    # 合并全国性假日
    national_holidays = holidays[holidays['locale'] == 'National']
    national_holidays = national_holidays[['date', 'type']].rename(columns={'type': 'holiday_type'})
    full_data = pd.merge(full_data, national_holidays, on='date', how='left')

    # 合并店铺元数据
    full_data = pd.merge(full_data, stores, on='store_nbr', how='left')

    # 合并交易量数据
    transactions = transactions.rename(columns={'transactions': 'daily_transactions'})
    full_data = pd.merge(full_data, transactions, on=['date', 'store_nbr'], how='left')

    # 填充缺失值
    # 油价是时间序列数据，通常具有连续性（相邻日期的油价变化不大），适合用前后填充
    full_data['oil_price'] = full_data['oil_price'].ffill().bfill()
    full_data['holiday_type'] = full_data['holiday_type'].fillna('No_Holiday')
    full_data['daily_transactions'] = full_data['daily_transactions'].fillna(0)

    return full_data



# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load
#
# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
#
# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
#
# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))
#
# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# --- 主流程 ---
if __name__ == "__main__":
    # 加载数据
    train, test, holidays, oil, stores, transactions = load_data()

    # 合并特征
    test['sales'] = np.nan  # 添加伪目标列
    full_data = merge_features(train, test, holidays, oil, stores, transactions)

    # 特征工程
    full_data = create_date_features(full_data)
    full_data = create_lag_features(full_data)
    full_data = create_rolling_features(full_data)

    # 新增衍生特征
    full_data['sales_per_transaction'] = full_data['sales'] / (full_data['daily_transactions'] + 1e-5)
    full_data['promo_intensity'] = full_data['onpromotion'] / full_data.groupby(['store_nbr', 'family'])['onpromotion'].transform('mean')
    full_data['oil_3day_ma'] = full_data['oil_price'].rolling(3).mean()

    # 标签编码
    #目的：将非数值型分类变量（如字符串或类别）转换为数值型，便于机器学习模型处理。
# 操作：对 family（商品类别）、holiday_type（假日类型）、city（城市）、state（州）四列进行标签编码。
#
# 标签编码会将每个唯一类别映射为一个整数（例如 ['A', 'B', 'C'] → [0, 1, 2]）。
    le = LabelEncoder()
    full_data['family'] = le.fit_transform(full_data['family'])
    full_data['holiday_type'] = le.fit_transform(full_data['holiday_type'])
    full_data['city'] = le.fit_transform(full_data['city'])
    full_data['state'] = le.fit_transform(full_data['state'])

    # 拆分数据集
    train = full_data[~full_data['sales'].isna()]
    test = full_data[full_data['sales'].isna()]

    # 定义特征列
    features = [
        'store_nbr', 'family', 'onpromotion', 'day', 'weekofyear', 'month', 'year',
        'dayofweek', 'is_weekend', 'is_month_start', 'is_month_end', 'quarter',
        'oil_price', 'oil_3day_ma', 'holiday_type', 'cluster', 'city', 'state',
        'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
        'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_28',
        'sales_per_transaction', 'promo_intensity', 'daily_transactions'
    ]

    # 数据分割
    X_train = train[train['date'] < '2017-06-15'][features]
    y_train = train[train['date'] < '2017-06-15']['sales']
    X_valid = train[train['date'] >= '2017-06-15'][features]
    y_valid = train[train['date'] >= '2017-06-15']['sales']

    # 目标值对数变换
    y_train_log = np.log1p(y_train)
    y_valid_log = np.log1p(y_valid)

    # 训练LightGBM
    train_data = lgb.Dataset(X_train, label=y_train_log, categorical_feature=['family', 'holiday_type'])
    valid_data = lgb.Dataset(X_valid, label=y_valid_log, categorical_feature=['family', 'holiday_type'])

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 128,  # 根据硬件调整
        'max_depth': 7,
        'feature_fraction': 0.8,
        'bagging_freq': 5,
        'verbosity': -1,
        'seed': 42
    }

    model = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, valid_data],
        num_boost_round=5000,
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(500)
        ]
    )

    # 验证集评估
    y_pred = np.expm1(model.predict(X_valid))
    print(f'Validation RMSLE: {np.sqrt(mean_squared_log_error(y_valid, y_pred)):.4f}')

    # 生成预测
    test_pred = np.expm1(model.predict(test[features]))
    submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')
    submission['unit_sales'] = np.maximum(0, test_pred)  # 确保非负
    submission.to_csv(OUTPUT_PATH + 'final_submission.csv', index=False)

    print("✅ 预测文件已生成：final_submission.csv")


# # --- Imports ---
# import pandas as pd
# import numpy as np
# import lightgbm as lgb
# from sklearn.metrics import mean_squared_log_error
# from sklearn.preprocessing import LabelEncoder
#
# # --- Load the Data ---
# train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
# test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
#
# # --- Feature Engineering Functions ---
# def create_date_features(df):
#     df['day'] = df['date'].dt.day
#     df['weekofyear'] = df['date'].dt.isocalendar().week.astype('int')
#     df['month'] = df['date'].dt.month
#     df['year'] = df['date'].dt.year
#     df['dayofweek'] = df['date'].dt.dayofweek
#     df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
#     df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
#     df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
#     return df
#
# def create_lag_features(df, lags=[1, 7, 14, 28]):
#     for lag in lags:
#         df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
#     return df
#
# def create_rolling_features(df, windows=[7, 14, 28]):
#     for window in windows:
#         df[f'sales_roll_mean_{window}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(1).rolling(window=window).mean()
#     return df
#
# # --- Label Encoding ---
# le = LabelEncoder()
# train['family'] = le.fit_transform(train['family'])
# test['family'] = le.transform(test['family'])
#
# # --- Merge train and test for feature engineering ---
# test['sales'] = np.nan  # Dummy sales column
# all_data = pd.concat([train, test], axis=0).sort_values(['store_nbr', 'family', 'date'])
#
# # --- Feature Engineering ---
# all_data = create_date_features(all_data)
# all_data = create_lag_features(all_data)
# all_data = create_rolling_features(all_data)
#
# # --- Split back ---
# train = all_data[~all_data['sales'].isna()]
# test = all_data[all_data['sales'].isna()]
#
# # --- Fill missing ---
# train.fillna(-1, inplace=True)
# test.fillna(-1, inplace=True)
#
# # --- Prepare Features and Target ---
# features = [
#     'store_nbr', 'family', 'onpromotion', 'day', 'weekofyear', 'month', 'year', 'dayofweek',
#     'is_weekend', 'is_month_start', 'is_month_end',
#     'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
#     'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_28'
# ]
#
# target = 'sales'
#
# # --- Train-Validation Split ---
# X_train = train[train['date'] < '2017-07-01'][features]
# y_train = train[train['date'] < '2017-07-01'][target]
# X_valid = train[train['date'] >= '2017-07-01'][features]
# y_valid = train[train['date'] >= '2017-07-01'][target]
#
# # --- Important Trick: Log Transform Target ---
# y_train_log = np.log1p(y_train)
# y_valid_log = np.log1p(y_valid)
#
# # --- LightGBM Dataset ---
# train_data = lgb.Dataset(X_train, label=y_train_log, categorical_feature=['family'])
# valid_data = lgb.Dataset(X_valid, label=y_valid_log, categorical_feature=['family'])
#
# # --- LightGBM Parameters ---
# params = {
#     'objective': 'regression',
#     'metric': 'rmse',
#     'boosting_type': 'gbdt',
#     'learning_rate': 0.02,
#     'num_leaves': 256,
#     'max_depth': 8,
#     'feature_fraction': 0.8,
#     'bagging_fraction': 0.8,
#     'bagging_freq': 5,
#     'lambda_l1': 1,
#     'lambda_l2': 1,
#     'seed': 42,
#     'verbosity': -1,
# }
#
# # --- Train the Model ---
# model = lgb.train(
#     params,
#     train_data,
#     valid_sets=[train_data, valid_data],
#     num_boost_round=20000,
#     callbacks=[
#         lgb.early_stopping(stopping_rounds=300),
#         lgb.log_evaluation(period=500)
#     ]
# )
#
# # --- Validation RMSLE ---
# y_pred_valid_log = model.predict(X_valid, num_iteration=model.best_iteration)
# y_pred_valid = np.expm1(y_pred_valid_log)  # Inverse transform
#
# def rmsle(y_true, y_pred):
#     return np.sqrt(mean_squared_log_error(y_true, np.maximum(0, y_pred)))
#
# print(f'Validation RMSLE: {rmsle(y_valid, y_pred_valid):.5f}')
#
# # --- Predict on Test Set ---
# X_test = test[features]
# y_test_pred_log = model.predict(X_test, num_iteration=model.best_iteration)
# y_test_pred = np.expm1(y_test_pred_log)  # Inverse transform
# y_test_pred = np.maximum(0, y_test_pred)
#
# # --- Submission ---
# submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
# submission['sales'] = y_test_pred
# submission.to_csv('submission.csv', index=False)
#
# print("✅ Submission file created successfully brother (Optimized)!")

Training until validation scores don't improve for 200 rounds
